# Packages to Import

In [1]:
import time
import math
import statistics
from typing import Iterable, Sequence

import numpy as np
import pandas as pd

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp

try:
    from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
    IBM_AVAILABLE = True
except Exception:
    IBM_AVAILABLE = False

# The Perceptron
The simplest perceptron is just a linear threshold function. Which means:\
$x \in \mathbb{R}^{d}$, choose wieghts $w \in \mathbb{R^d}$ and a bias $b \in \mathbb{R}$ and we define:\
$$f(x) = 
     \begin{cases}
       \text{1,} &\quad\text{if $w^Tx + b >0$ }\\
       \text{0,} &\quad\text{otherwise} \\
     \end{cases}
$$
My previous implementation of a simple perceptron for CIUFLA 2025 was the code present in the second code cell bellow.

In [2]:
class Perceptron:
    def __init__(self, n_features: int):
        self.w = [0.0] * n_features
        self.b = 0.0

    def score(self, x: Sequence[float]) -> float:
        return sum(wi * xi for wi, xi in zip(self.w, x)) + self.b

    def predict(self, x: Sequence[float]) -> int:
        return 1 if self.score(x) > 0 else -1

    def fit(self, X: Iterable[Sequence[float]], y: Iterable[int], epochs: int = 10) -> None:
        X = list(X)
        y = list(y)
        for _ in range(epochs):
            for xi, yi in zip(X, y):
                if yi * self.score(xi) <= 0:
                    self.w = [wi + yi * xij for wi, xij in zip(self.w, xi)]
                    self.b += yi

# A Quantum Perceptron
So now, given the motivation arised by reading the Turing paper, what follows is my implementation of a perceptron in a quantum computer.

In [3]:
class QPerceptron:
    """
    1-qubit quantum wrapper around the affine score:
        s = w·x + b
        theta = clip(s, -pi/2, pi/2)
        prepare Ry(theta)|0>
        score = <X>
        predict = sign(<X>)
    """

    def __init__(self, n_features: int):
        self.w = [0.0] * n_features
        self.b = 0.0
        self.observable = SparsePauliOp.from_list([("X", 1.0)])

    def linear_score(self, x: Sequence[float]) -> float:
        return sum(wi * xi for wi, xi in zip(self.w, x)) + self.b

    def _angle(self, x: Sequence[float]) -> float:
        s = self.linear_score(x)
        return max(-math.pi / 2, min(math.pi / 2, s))

    def _circuit(self, x: Sequence[float]) -> QuantumCircuit:
        qc = QuantumCircuit(1)
        qc.ry(self._angle(x), 0)
        return qc

    def quantum_score(self, x: Sequence[float]) -> float:
        qc = self._circuit(x)
        psi = Statevector.from_instruction(qc)
        return float(psi.expectation_value(self.observable).real)

    def predict(self, x: Sequence[float]) -> int:
        return 1 if self.quantum_score(x) > 0 else -1

    def predict_from_angle_only(self, x: Sequence[float]) -> int:
        value = math.sin(self._angle(x))
        return 1 if value > 0 else -1

    def fit(self, X: Iterable[Sequence[float]], y: Iterable[int], epochs: int = 10) -> None:
        X = list(X)
        y = list(y)
        for _ in range(epochs):
            for xi, yi in zip(X, y):
                if yi * self.linear_score(xi) <= 0:
                    self.w = [wi + yi * xij for wi, xij in zip(self.w, xi)]
                    self.b += yi


class QPerceptronIBM(QPerceptron):
    """
        Same model, but expectation values are evaluated with IBM Runtime Estimator.
        Authentication must be passed explicitly via a service object.
    """

    def __init__(self, n_features: int):
        super().__init__(n_features)

    def quantum_scores_ibm(
        self,
        X: Iterable[Sequence[float]],
        service,
        backend=None,
        optimization_level: int = 1,
    ):
        if not IBM_AVAILABLE:
            raise RuntimeError("qiskit-ibm-runtime is not installed.")

        X = list(X)

        if backend is None:
            backend = service.least_busy(operational=True, simulator=False)

        circuits = [self._circuit(x) for x in X]

        t0 = time.perf_counter()
        pm = generate_preset_pass_manager(
            backend=backend,
            optimization_level=optimization_level,
        )
        isa_circuits = [pm.run(qc) for qc in circuits]
        t1 = time.perf_counter()

        isa_observables = [self.observable.apply_layout(qc.layout) for qc in isa_circuits]

        estimator = Estimator(mode=backend)

        t2 = time.perf_counter()
        job = estimator.run(list(zip(isa_circuits, isa_observables)))
        result = job.result()
        t3 = time.perf_counter()

        values = [float(pub.data.evs) for pub in result]

        timing = {
            "backend": backend.name,
            "n_circuits": len(X),
            "transpile_s": t1 - t0,
            "submit_wait_s": t3 - t2,
            "total_s": (t1 - t0) + (t3 - t2),
            "per_sample_s": ((t1 - t0) + (t3 - t2)) / max(len(X), 1),
        }
        return values, timing

    def predict_ibm(
        self,
        X: Iterable[Sequence[float]],
        service,
        backend=None,
        optimization_level: int = 1,
    ):
        vals, timing = self.quantum_scores_ibm(
            X=X,
            service=service,
            backend=backend,
            optimization_level=optimization_level,
        )
        preds = [1 if v > 0 else -1 for v in vals]
        return preds, vals, timing

# Running benchmarks and tests

In [4]:
LOCAL_SAMPLE_SIZES = (100, 1000, 5000)
LOCAL_FEATURE_SIZES = (2, 10, 50)
EPOCHS = 10
REPEAT = 10

RUN_IBM = False
IBM_API_KEY = "API"
IBM_INSTANCE = "INSTANCE"
IBM_BACKEND_NAME = "BACKEND"
IBM_N_SAMPLES = 4
IBM_N_FEATURES = 2
IBM_OPTIMIZATION_LEVEL = 1

In [5]:
def make_dataset(n_samples: int, n_features: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n_samples, n_features))
    true_w = rng.normal(size=n_features)
    true_b = rng.normal()
    y = np.sign(X @ true_w + true_b)
    y[y == 0] = 1
    return X.tolist(), y.astype(int).tolist()

def time_fn(fn, repeat: int = 10) -> dict:
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return {
        "median": statistics.median(times),
        "mean": statistics.mean(times),
        "min": min(times),
        "max": max(times),
        "all": times,
    }

def match_rate(a, b) -> float:
    return sum(int(x == y) for x, y in zip(a, b)) / len(a)

def benchmark_local(sample_sizes=(100, 1000, 5000), feature_sizes=(2, 10, 50), epochs=10, repeat=10):
    rows = []

    for n in sample_sizes:
        for d in feature_sizes:
            X, y = make_dataset(n, d, seed=42)

            p = Perceptron(d)
            qp = QPerceptron(d)

            fit_p = time_fn(lambda: p.fit(X, y, epochs=epochs), repeat=repeat)
            fit_q = time_fn(lambda: qp.fit(X, y, epochs=epochs), repeat=repeat)

            p = Perceptron(d)
            qp = QPerceptron(d)
            p.fit(X, y, epochs=epochs)
            qp.fit(X, y, epochs=epochs)

            pred_p = time_fn(lambda: [p.predict(x) for x in X], repeat=repeat)
            pred_q = time_fn(lambda: [qp.predict(x) for x in X], repeat=repeat)
            pred_angle = time_fn(lambda: [qp.predict_from_angle_only(x) for x in X], repeat=repeat)

            classical_preds = [p.predict(x) for x in X]
            quantum_preds = [qp.predict(x) for x in X]
            angle_preds = [qp.predict_from_angle_only(x) for x in X]

            rows.append({
                "n_samples": n,
                "n_features": d,
                "classical_fit_median_s": fit_p["median"],
                "quantum_fit_median_s": fit_q["median"],
                "classical_predict_median_s": pred_p["median"],
                "angle_only_predict_median_s": pred_angle["median"],
                "full_quantum_predict_median_s": pred_q["median"],
                "fit_slowdown": fit_q["median"] / fit_p["median"] if fit_p["median"] > 0 else float("inf"),
                "angle_only_slowdown": pred_angle["median"] / pred_p["median"] if pred_p["median"] > 0 else float("inf"),
                "full_quantum_slowdown": pred_q["median"] / pred_p["median"] if pred_p["median"] > 0 else float("inf"),
                "classical_vs_quantum_agreement": match_rate(classical_preds, quantum_preds),
                "classical_vs_angle_agreement": match_rate(classical_preds, angle_preds),
            })

    return pd.DataFrame(rows)

def get_backend_by_name(service, backend_name: str):
    matches = [b for b in service.backends() if b.name == backend_name]
    if not matches:
        raise ValueError(f"Backend '{backend_name}' not found. Available: {[b.name for b in service.backends()]}")
    return matches[0]

In [6]:
df_local = benchmark_local(
    sample_sizes=LOCAL_SAMPLE_SIZES,
    feature_sizes=LOCAL_FEATURE_SIZES,
    epochs=EPOCHS,
    repeat=REPEAT,
)

print("=== LOCAL BENCHMARK ===")
display(df_local)

if RUN_IBM:
    if not IBM_AVAILABLE:
        raise RuntimeError("RUN_IBM=True, but qiskit-ibm-runtime is not installed.")

    if IBM_API_KEY == "YOUR_IBM_API_KEY":
        raise ValueError("Set IBM_API_KEY before running hardware execution.")

    print("\n=== IBM HARDWARE RUN ===")
    QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token=IBM_API_KEY, instance=IBM_INSTANCE)
    service = QiskitRuntimeService()

    backend = get_backend_by_name(service, IBM_BACKEND_NAME)

    X_ibm, y_ibm = make_dataset(IBM_N_SAMPLES, IBM_N_FEATURES, seed=7)

    p_ibm = Perceptron(IBM_N_FEATURES)
    qp_ibm = QPerceptronIBM(IBM_N_FEATURES)

    p_ibm.fit(X_ibm, y_ibm, epochs=EPOCHS)
    qp_ibm.fit(X_ibm, y_ibm, epochs=EPOCHS)

    classical_preds = [p_ibm.predict(x) for x in X_ibm]

    q_preds, q_vals, q_timing = qp_ibm.predict_ibm(
        X=X_ibm,
        service=service,
        backend=backend,
        optimization_level=IBM_OPTIMIZATION_LEVEL,
    )

    df_ibm = pd.DataFrame([{
        "backend": q_timing["backend"],
        "n_samples": q_timing["n_circuits"],
        "transpile_s": q_timing["transpile_s"],
        "submit_wait_s": q_timing["submit_wait_s"],
        "total_s": q_timing["total_s"],
        "per_sample_s": q_timing["per_sample_s"],
        "match_rate_vs_classical": match_rate(classical_preds, q_preds),
    }])

    print("IBM input samples:")
    for i, (x, y_true) in enumerate(zip(X_ibm, y_ibm), start=1):
        print(f"{i:02d}. x={x}, y={y_true}")

    print("\nClassical predictions:")
    print(classical_preds)

    print("\nIBM quantum predictions:")
    print(q_preds)

    print("\nIBM expectation values:")
    print(q_vals)

    print("\nIBM timing summary:")
    display(df_ibm)

=== LOCAL BENCHMARK ===


,n_samples,n_features,classical_fit_median_s,quantum_fit_median_s,classical_predict_median_s,angle_only_predict_median_s,full_quantum_predict_median_s,fit_slowdown,angle_only_slowdown,full_quantum_slowdown,classical_vs_quantum_agreement,classical_vs_angle_agreement
0,100,2,0.000357,0.000333,0.000034,0.000049,0.012725,0.932967,1.456566,378.034377,1.0,1.0
1,100,10,0.000601,0.000631,0.000061,0.000069,0.012213,1.048874,1.134252,201.396995,1.0,1.0
2,100,50,0.001704,0.001722,0.000165,0.000193,0.012377,1.010725,1.168862,75.085688,1.0,1.0
3,1000,2,0.003397,0.003398,0.000332,0.000482,0.124848,1.000072,1.453608,376.378448,1.0,1.0
4,1000,10,0.006078,0.006084,0.000594,0.000741,0.113719,1.001002,1.248555,191.523973,1.0,1.0
5,1000,50,0.018203,0.017263,0.001718,0.001799,0.118251,0.948369,1.046970,68.811605,1.0,1.0
6,5000,2,0.015868,0.015814,0.001538,0.002218,0.586959,0.996569,1.442633,381.686947,1.0,1.0
7,5000,10,0.029274,0.030126,0.002964,0.003391,0.570712,1.029104,1.143995,192.541029,1.0,1.0
8,5000,50,0.087048,0.087213,0.008450,0.009678,0.585752,1.001902,1.145357,69.319209,1.0,1.0


# Notes on Quantum Perceptrons

Classical perceptrons [1] are linear classifiers: they compute a weighted sum of inputs and apply a threshold. Quantum perceptrons [6] map inputs into quantum states and perform unitary transformations followed by measurement. In many cases, the results are equivalent to classical perceptrons because both rely on linear algebra operations in vector (Hilbert) spaces. True quantum advantage typically requires exploiting entanglement, feature-space embeddings [7,8], or variational circuits [4,9]. Recent work shows even a single qubit can solve non-trivial classical tasks [10,11]. From my exploratory analysis, there are some gaps I need to fill in my quantum information/quantum computing knowledge.

**Next steps:**
1. Study Wiebe et al. [6] for quantum perceptron construction.
2. Implement small quantum perceptrons and compare with classical outcomes.
3. Explore scenarios for potential quantum advantage using feature embeddings or variational circuits.

## References

[1] F. Rosenblatt, *The perceptron: A probabilistic model for information storage and organization in the brain*, Psych. Rev., 65(6):386–408, 1958.  
[2] J. Biamonte et al., *Quantum machine learning*, Nature, 549(7671):195–202, 2017.  
[3] M. Schuld et al., *An introduction to quantum machine learning*, Contemp. Phys., 56(2):172–185, 2015.  
[4] M. Cerezo et al., *Variational quantum algorithms*, Nat. Rev. Phys., 3:625–644, 2021.  
[5] M. Nielsen & I. Chuang, *Quantum Computation and Quantum Information*, 2010.  
[6] N. Wiebe et al., *Quantum perceptron models*, arXiv:1602.04799, 2016.  
[7] M. Schuld & N. Killoran, *Quantum machine learning in feature Hilbert spaces*, PRL, 122(4):040504, 2019.  
[8] V. Havlíček et al., *Supervised learning with quantum-enhanced feature spaces*, Nature, 567:209–212, 2019.  
[9] M. Schuld et al., *Circuit-centric quantum classifiers*, PRA, 101(3):032308, 2020.  
[10] M. P. Cuéllar, *What we can do with one qubit in quantum machine learning*, Quantum Mach. Intel., 6:76, 2024.  
[11] I. V. Grossu, *Single qubit neural quantum circuit for solving exclusive-or*, MethodsX, 8:101573, 2021.